# A3e — build the reviewer's hypothesis, and hand it back to the detector

> **Reviewer #2, major point 9.** *"The authors should consider whether the detected granules could
> instead reflect locally elevated ambient RNA rather than genuine subcellular compartments."*

`A3d_local_null.ipynb` answers this statistically: hold each granule's transcript count fixed,
redraw its contents from the RNA immediately around it, and the observed neuronal enrichment and
glial depletion sit about a hundred null standard deviations away. That is an answer in z-scores.

This notebook answers the same question with **the detector itself**, which is what the reviewer
literally asked for. Take real granules. Keep every transcript exactly where it is. Replace only
the **gene identities**, drawing them from the ambient RNA around that granule. Then re-run
mcDETECT over the whole section.

- If mcDETECT is calling locally dense patches of ambient RNA, it calls these back.
- If it is calling compositionally distinct compartments, it misses them.

**One draw, one detection run per section.** Nothing here is repeated or averaged over replicates —
the answer is a re-detection rate, read against two controls, not a null distribution.

## Why this is not circular

mcDETECT seeds DBSCAN on the 20 granule markers, so *"strip the markers and it stops detecting"*
would prove nothing on its own. Three things make the number informative.

1. **Local density is held exactly fixed.** Every transcript keeps its own `(x, y, z)`; only
   `target` changes. The reviewer's *"locally elevated ambient RNA"* is preserved in full, and
   composition is the only thing that varies.
2. **Ambient is not marker-free.** The 20 markers are roughly a third of all transcripts, so a
   local ambient draw puts real marker labels back into the pseudo-granule. The re-detection rate
   is a **measured** quantity that could have come out high.
3. **A control arm separates composition from geometry.** Relabelling also scrambles *which* point
   carries which gene, and that alone could break DBSCAN's ε-connectivity. A second arm scrambles
   in exactly the same way while preserving the granule's own composition.

## The three arms

| arm | what changes | what it tells us |
| --- | --- | --- |
| `ambient` | labels redrawn from the residual extrasomatic RNA within 5 µm of the granule centre | the hypothesis |
| `scramble` | the granule's **own** labels permuted among its own points | the machinery control — composition preserved exactly |
| `untouched` | nothing | the load-bearing control: must come back at ~100% |

`ambient` read against `scramble` is the compositional effect on its own. `untouched` is what makes
a single detection run sufficient — it is simultaneously the proof that the re-run reproduces the
published pipeline and the proof that the perturbation stayed local.

## Not a repeat of A3b

`A3b_vicinity.ipynb` *displaces* a sphere to a nearby empty location and applies a detectability
predicate. A3e keeps the location, changes the contents, and runs the real detector end to end.

## How to run this

**Run this notebook twice, around the HGCC job** — not because anything is sampled twice, but
because the detection happens on a compute node in between.

1. **First run, sections 0–5, locally.** Writes `output/a3e/a3e_relabel_<sample>.parquet`, the
   patch. Sections 6–7 print "waiting for the detection run" and the notebook still completes.
2. `scp output/a3e/` to HGCC, `sbatch slurm/run_pseudo_detection.sh` (2 tasks, ~4 h WT / ~2 h AD),
   then rsync `output/a3e/detect_<sample>/` back.
3. **Second run, top to bottom.** Section 4 reads the patch it already wrote rather than redrawing
   it, so the analysis is unchanged; sections 6–7 light up.

Run from `R2_revision/ambient_controls/`, on the `mcDETECT-env` kernel.

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

sys.path.insert(0, str(Path.cwd()))          # run this notebook from ambient_controls/
import a3_config as C
import a3_common as A3

warnings.filterwarnings("ignore")

# -------------------- runtime gates -------------------- #
# THE DEFAULTS BELOW PRODUCE THE FINAL TABLES. Run top to bottom and change nothing.
OVERWRITE = False        # True -> rebuild the patch even if one is already on disk
RUN_PRECHECK = True      # section 5. Indicative only, and the expensive local step: it builds one
                         #   KD-tree per marker gene over ~30 M transcripts. The detection run is
                         #   the authoritative answer; this only tells you early what it will say.
VALIDATE = True          # section 7 correctness gates

ARMS = list(C.PSEUDO_ARMS)
UNTOUCHED = ARMS.index("untouched")
RADIUS = C.PSEUDO_POOL_RADIUS

C.ensure_dirs()
OUT = C.A3E_DIR
MARKERS = list(C.SYN_GENES)

print("writing to      :", OUT)
print(f"arms            : {ARMS}, {C.PSEUDO_FRAC:.0%} of granules per converted arm")
print(f"ambient pool    : residual extrasomatic RNA within {RADIUS:g} um of the granule centre, "
      f"all z pooled (area {np.pi * RADIUS ** 2:.1f} um^2)")
print(f"retention       : pool >= max({C.PSEUDO_MIN_POOL}, the granule's own transcript count), "
      f"applied to EVERY arm")
print(f"radius ladder   : {C.PSEUDO_POOL_RADIUS_LADDER} um  (reported in section 3, never drawn from)")
print(f"markers         : {len(MARKERS)} (the genes DBSCAN seeds on)")
print(f"matching        : {C.PSEUDO_MATCH_CRITERIA}, primary {C.PSEUDO_MATCH_PRIMARY!r}")
print(f"control gate    : untouched arm must be re-detected at >= {C.PSEUDO_CONTROL_MIN:.0%}")

have_detection = {s: (C.pseudo_detect_dir(s) / "spheres.parquet").exists() for s in C.SAMPLES}
print("\ndetection output:", {s: ("present" if v else "not yet -- first run")
                              for s, v in have_detection.items()})

writing to      : /Users/chenyang/Desktop/mcDETECT/R2_revision/ambient_controls/output/a3e
arms            : ['ambient', 'scramble', 'untouched'], 10% of granules per converted arm
ambient pool    : residual extrasomatic RNA within 5 um of the granule centre, all z pooled (area 78.5 um^2)
retention       : pool >= max(50, the granule's own transcript count), applied to EVERY arm
radius ladder   : [4.0, 5.0, 6.0, 7.0] um  (reported in section 3, never drawn from)
markers         : 20 (the genes DBSCAN seeds on)
matching        : ['center_in', 'intersect', 'merge'], primary 'center_in'
control gate    : untouched arm must be re-detected at >= 99%

detection output: {'WT': 'not yet -- first run', 'AD': 'not yet -- first run'}


## 1. The granules, and the three arms

The population is the **published Set 2** granules — `output/<dataset>/granules.parquet`, the
granules the paper reports. That is deliberate: the reviewer's question is about the granules we
published, so those are the ones that get converted, and the re-detection reproduces the published
pipeline rather than a variant of it.

Two disjoint tenths are drawn in a single call, so no granule can land in both arms. The seed is
crc32 of the sample name — never `hash()`, whose string hashing is salted per process and would
give a different sample on every kernel restart.

In [2]:
gran, arms_of = {}, {}
rows = []
for s in C.SAMPLES:
    g = pd.read_parquet(C.mcdetect_granules_path(s))
    rng = np.random.default_rng(zlib.crc32(f"{C.PSEUDO_SEED}|{s}".encode()))
    code_ = A3.sample_pseudo_arms(len(g), rng)
    gran[s], arms_of[s] = g, code_
    rows.append(dict(sample=s, n_granules=len(g),
                     **{f"n_{a}": int((code_ == i).sum()) for i, a in enumerate(ARMS)}))

arm_counts = pd.DataFrame(rows)
print(arm_counts.to_string(index=False))
for s in C.SAMPLES:
    print(f"\n[{s}] published granules {len(gran[s]):,}; "
          f"sphere_r median {gran[s]['sphere_r'].median():.2f} um, "
          f"size median {gran[s]['size'].median():.0f} transcripts")

sample  n_granules  n_ambient  n_scramble  n_untouched
    WT      681337      68134       68134       545069
    AD      398809      39881       39881       319047

[WT] published granules 681,337; sphere_r median 0.93 um, size median 4 transcripts

[AD] published granules 398,809; sphere_r median 0.95 um, size median 4 transcripts


## 2. Which transcripts belong to each converted granule

Reuses A3c's cached per-transcript compartment labels — one `int8` column in the transcript
table's own row order — so the 10⁸-transcripts-against-10⁶-spheres assignment is not repeated. Only
the **granule layer** enters: transcripts inside a published sphere that do *not* overlap a
nucleus. That means the KD-tree is built over ~6 M points rather than 10⁸, and it means the
relabelling never touches somatic RNA. Intrasomatic transcripts that happen to fall inside a
granule sphere are left exactly as they are — they belong to the somatic layer by A3c's partition,
`in_soma_thr = 0.1` caps them at a tenth of any sphere, and rewriting them would be modifying soma
content to answer a question about extrasomatic RNA.

**The buffer is the trap here.** The containment query runs at `sphere_r + KG_BUFFER`, matching
`partition_transcripts`. `sphere_r` is the *minimum-enclosing* radius, so a granule's own support
points — overwhelmingly its seed gene — sit exactly on the sphere surface. A bare-radius query
loses 11.6% of the granule layer and 93.6% of what it loses are markers. Elsewhere in A3 that would
merely bias a comparison; here it would be fatal, because the unrelabelled seed transcripts would
stay in place and the pseudo-granule would be re-detected for free.

Spheres overlap, so a transcript can fall inside several. `granule_members` assigns each to the
nearest centre, ties by the lower granule row, so every transcript is rewritten exactly once and
the two converted arms stay disjoint at transcript level.

**Untouched granules that share transcripts with a converted one are contaminated** — part of their
content was rewritten on another granule's behalf. They are removed from the control set and
counted, not left in to flatter it.

In [ ]:
members, layer_rows, tx_len, contam, all_targets = {}, {}, {}, {}, {}
owner_all, k_all, layer_xyz = {}, {}, {}     # full-granule ownership, for section 6
for s in C.SAMPLES:
    # target is read in this same pass: the granule layer's current labels are all sections 4 and 6
    # need from it, and its category list doubles as the full target vocabulary (290 panel genes
    # plus the Blank probes). Section 3 re-reads coordinates for the RESIDUAL layer separately, one
    # sample at a time, so no more than one section's worth is ever resident.
    tx = A3.load_transcripts(s, columns=["global_x", "global_y", "global_z", "target"])
    all_targets[s] = list(tx["target"].cat.categories)
    lab = pd.read_parquet(C.transcript_layer_path(s))["layer"].to_numpy()
    assert len(lab) == len(tx), (
        f"[{s}] layer cache has {len(lab):,} rows against {len(tx):,} transcripts. The cache is "
        f"POSITIONAL, so a mismatch means it was written for a different table.")
    tx_len[s] = len(tx)

    gl = np.flatnonzero(lab == C.DE_LAYERS.index("granule"))       # positional rows, granule layer
    pts = tx[["global_x", "global_y", "global_z"]].to_numpy(dtype=float)[gl]
    tcodes = tx["target"].cat.codes.to_numpy()[gl]                 # current labels, granule layer
    layer_rows[s] = gl
    del tx, lab

    conv = np.flatnonzero(arms_of[s] != UNTOUCHED)
    print(f"[{s}] granule layer {len(gl):,} transcripts; "
          f"assigning {len(conv):,} converted spheres", flush=True)
    m = A3.granule_members(gran[s].iloc[conv].reset_index(drop=True), pts)
    m["granule"] = conv[m["granule"].to_numpy()]                   # back to published-row indices
    m["row"] = gl[m["point"].to_numpy()]                           # back to transcript-table rows
    m["cur_code"] = tcodes[m["point"].to_numpy()]                  # into all_targets[s]
    members[s] = m

    # untouched granules that share any rewritten transcript
    unt = np.flatnonzero(arms_of[s] == UNTOUCHED)
    tree = cKDTree(pts[m["point"].to_numpy()])
    n_hit = tree.query_ball_point(
        gran[s].iloc[unt][["sphere_x", "sphere_y", "layer_z"]].to_numpy(dtype=float),
        gran[s].iloc[unt]["sphere_r"].to_numpy(dtype=float) + C.KG_BUFFER,
        workers=-1, return_length=True)
    contam[s] = unt[np.asarray(n_hit) > 0]
    del tree, tcodes

    # A SECOND, SEPARATE ownership pass, over ALL granules -- section 6's provenance criterion
    # needs to know whose transcripts a re-detected sphere sits on, including the untouched
    # control's. It must NOT replace `m`: granule_members assigns to the nearest granule AMONG
    # THOSE PASSED IN, so recomputing `m` over all 681 K would move transcripts to untouched
    # neighbours and desynchronise the gates from the patch already written. Two maps, one job
    # each. This is consistent, because every granule-layer transcript inside ANY converted sphere
    # was relabelled whichever converted granule claimed it -- so a converted granule's provenance
    # set is always a subset of what was rewritten.
    ma = A3.granule_members(gran[s], pts)
    assert len(ma) == len(gl), (
        f"[{s}] {len(gl) - len(ma):,} granule-layer transcripts fall in no sphere -- the layer "
        f"cache and the granule table disagree")
    owner_all[s] = np.empty(len(gl), dtype=np.int64)
    owner_all[s][ma["point"].to_numpy()] = ma["granule"].to_numpy()
    k_all[s] = np.bincount(ma["granule"].to_numpy(), minlength=len(gran[s]))
    layer_xyz[s] = pts
    print(f"[{s}] full ownership: {int((k_all[s] > 0).sum()):,}/{len(gran[s]):,} granules own at "
          f"least one transcript (median {np.median(k_all[s][k_all[s] > 0]):.0f})", flush=True)
    del ma

    per = m.groupby("granule").size()
    print(f"[{s}] {len(m):,} transcripts to rewrite over {per.size:,} spheres "
          f"(median {per.median():.0f}, max {per.max():.0f}); "
          f"{len(contam[s]):,} untouched granules contaminated "
          f"({len(contam[s]) / max(len(unt), 1):.2%}) and excluded from the control")

## 3. The local ambient pool, and the draw

For each granule, the pool is the **residual extrasomatic RNA within `PSEUDO_POOL_RADIUS` = 5 µm of
its centre**, pooling all seven z-planes.

**Why a disc and not a square of A3d's grid.** The obvious implementation reuses A3d's 10 µm
lattice and gives each granule the residual RNA of whichever square its centre happens to land in.
That was the first version of this notebook and it is the wrong instrument here: the granule is not
centred in that square. Its centre sits a median ~2.5 µm — and up to 7 µm at a corner — from the
middle of the neighbourhood it is being compared against. A3d can live with that because it needs a
*partition* of the section to sum closed-form moments over. A3e needs no partition at all; it needs
one neighbourhood per granule, and a disc states that literally. It also removes the 3 × 3 fallback:
a disc sized in µm cannot be "too thin" the way an arbitrary lattice square can.

The disc is **21% tighter in area** than A3d's square (78.5 against 100 µm²), so the locality claim
here is at least as strong as A3d's.

**Only the residual extrasomatic layer enters.** That layer already excludes *every* granule's
transcripts, which satisfies "excluding the granule's own transcripts" more strongly than asked: a
granule draws from RNA that no granule was built from, its own included. And the pool is over **all
targets** — 290 panel genes plus the Blank probes — not A3d's 252 neutral genes. Restricting it to
neutral genes would remove every marker from the pool and make non-detection true by construction,
which is the exact circularity this notebook exists to avoid.

### The retention rule, and the ladder that checks it

A granule is redrawn only if its own neighbourhood can actually supply the draw:

```
pool_size  >=  max(PSEUDO_MIN_POOL, k)        k = the granule's own transcript count
```

The `k` half is what makes the without-replacement draw well defined. Without it, a granule larger
than its own surroundings would have to be drawn *with* replacement — a different sampling scheme,
applied to exactly the largest granules, which are also the easiest to re-detect.
`draw_local_ambient` asserts the rule rather than falling back, so a violation stops the run.

The rule is applied to **every arm**, not just `ambient`. Neighbourhood density plausibly predicts
how re-detectable a granule is, so restricting only the arm that draws from the pool would confound
`ambient` against `scramble` with local density. Granules that fail are labelled
`excluded_thin_pool` and reported separately; they are never folded into an arm.

The cell below scores the whole radius ladder first — counts only, no neighbour lists, so a whole
ladder is cheap — and prints retention per radius. **If retention at 5 µm is poor, raise
`PSEUDO_POOL_RADIUS` and re-run this cell.** Nothing is written to disk until section 4.

It then draws, in the same per-sample loop, so that only one section's residual tree
(~68 M points for WT) is ever resident. `ambient` granules draw their new gene identities here;
`scramble` needs no pool and is permuted in section 4.

In [4]:
pool_size, pool_keep, drawn_codes, ladder = {}, {}, {}, []
_cached = {s: C.pseudo_relabel_path(s).exists() and not OVERWRITE for s in C.SAMPLES}

for s in C.SAMPLES:
    # Residual coordinates for THIS sample only, then freed: 68 M rows is ~1.1 GB and there is no
    # reason for both sections to be resident at once. The draw happens inside this same loop for
    # the same reason.
    tx = A3.load_transcripts(s, columns=["global_x", "global_y", "target"])
    assert list(tx["target"].cat.categories) == all_targets[s], "target category order drifted"
    lab = pd.read_parquet(C.transcript_layer_path(s))["layer"].to_numpy()
    res = np.flatnonzero(lab == C.DE_LAYERS.index("residual_extrasomatic"))
    res_xy = tx[["global_x", "global_y"]].to_numpy(dtype=float)[res]
    res_code = tx["target"].cat.codes.to_numpy()[res].astype(np.int16)
    del tx, lab, res

    print(f"[{s}] building the 2-D tree over {len(res_xy):,} residual transcripts...", flush=True)
    tree = cKDTree(res_xy)
    del res_xy

    cen = gran[s][["sphere_x", "sphere_y"]].to_numpy(dtype=float)
    k_of = np.bincount(members[s]["granule"].to_numpy(), minlength=len(gran[s]))
    need = np.maximum(C.PSEUDO_MIN_POOL, k_of)          # the retention rule, per granule
    conv = arms_of[s] != UNTOUCHED

    # ---- the ladder: counts only, converted granules only, nothing drawn ----
    for r in C.PSEUDO_POOL_RADIUS_LADDER:
        n = A3.local_pool_sizes(cen[conv], tree, r, verbose=False)
        ladder.append(dict(sample=s, radius_um=r, n_granules=int(conv.sum()),
                           pool_median=float(np.median(n)), pool_q05=float(np.quantile(n, .05)),
                           n_retained=int((n >= need[conv]).sum()),
                           retention=float((n >= need[conv]).mean()),
                           chosen=bool(r == RADIUS)))

    # ---- the chosen radius, scored for EVERY granule so all three arms share one criterion ----
    sz = A3.local_pool_sizes(cen, tree, RADIUS, verbose=False)
    pool_size[s], pool_keep[s] = sz, sz >= need
    print(f"[{s}] at r={RADIUS:g} um: pool median {np.median(sz):,.0f}; retained "
          f"{pool_keep[s].sum():,}/{len(sz):,} granules ({pool_keep[s].mean():.2%}); "
          f"{int((~pool_keep[s] & conv).sum()):,} converted granules fail and become "
          f"excluded_thin_pool", flush=True)

    # ---- the ambient draw, here rather than in section 4 so the tree can be freed now ----
    if _cached[s]:
        print(f"[{s}] a patch is already on disk and OVERWRITE is False -- not redrawing")
    else:
        amb = np.flatnonzero((arms_of[s] == ARMS.index("ambient")) & pool_keep[s] & (k_of > 0))
        rng = np.random.default_rng(zlib.crc32(f"{C.PSEUDO_SEED}|draw|{s}".encode()))
        owner, code = A3.draw_local_ambient(cen[amb], tree, res_code, RADIUS, k_of[amb], rng,
                                            verbose=False)
        drawn_codes[s] = dict(granule=amb, owner=owner, code=code, k=k_of[amb])
        print(f"[{s}] drew {len(code):,} new identities for {len(amb):,} ambient granules")
    del tree, res_code

ladder = pd.DataFrame(ladder)
ladder.to_csv(OUT / "a3e_pool_ladder.csv", index=False)
print(f"\nradius ladder (converted granules; retention = pool >= max({C.PSEUDO_MIN_POOL}, "
      f"the granule's own transcript count)):")
print(ladder.to_string(index=False))

[WT] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_WT_1/processed_data/transcripts.parquet


[WT] 103,398,068 transcripts
[WT] building the 2-D tree over 68,430,447 residual transcripts...
[WT] at r=5 um: pool median 329; retained 680,778/681,337 granules (99.92%); 109 converted granules fail and become excluded_thin_pool
[WT] a patch is already on disk and OVERWRITE is False -- not redrawing
[AD] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_AD_1/processed_data/transcripts.parquet
[AD] 68,876,647 transcripts
[AD] building the 2-D tree over 44,297,301 residual transcripts...
[AD] at r=5 um: pool median 193; retained 396,778/398,809 granules (99.49%); 506 converted granules fail and become excluded_thin_pool
[AD] a patch is already on disk and OVERWRITE is False -- not redrawing

radius ladder (converted granules; retention = pool >= max(50, the granule's own transcript count)):
sample  radius_um  n_granules  pool_median  pool_q05  n_retained  retention  chosen
    WT        4.0      136268        215.0     105.0      135851   0.996940   False
    WT        5.0      13

## 4. The patch

**`ambient`** — drawn in section 3: each retained granule keeps the number of transcripts it
actually holds and takes that many gene identities, without replacement, from the residual RNA in
its own 5 µm disc.

**`scramble`** — each granule's own labels are permuted among its own points. The multiset of genes
is preserved exactly; only the assignment of gene to position changes. This is the control that
makes the `ambient` number readable, because both arms disturb positions in precisely the same way
and only `ambient` also changes what the granule is made of. It draws from no pool, but it is held
to the **same retention rule**, so the two arms cover matched populations rather than differing by
neighbourhood density.

Granules that fail the retention rule, or hold no granule-layer transcripts at all, are dropped
from their arm and counted.

What gets written is a **patch**, not a copy of the transcript table: the positional row index and
the new target, for the rewritten rows only. It is what actually travels to HGCC, and it is the
auditable record of exactly what was changed. `a3e_relabel_scope.csv` records the table length it
was built against, and `run_pseudo_detection.py` refuses to apply it to a table of any other
length.

In [5]:
patches, audit = {}, []
for s in C.SAMPLES:
    if _cached[s]:
        patches[s] = pd.read_parquet(C.pseudo_relabel_path(s))
        print(f"[{s}] read cached patch: {len(patches[s]):,} rows")
        continue

    rng = np.random.default_rng(zlib.crc32(f"{C.PSEUDO_SEED}|scramble|{s}".encode()))
    targets = np.asarray(all_targets[s])
    m = members[s]
    g = m["granule"].to_numpy()
    cur_code = m["cur_code"].to_numpy().astype(np.int32)
    arm = arms_of[s]
    keep = pool_keep[s][g]                       # the same rule for both converted arms

    out_rows, out_code = [], []
    for name in C.PSEUDO_CONVERTED_ARMS:
        sel = keep & (arm[g] == ARMS.index(name))
        gs, rs, cs = g[sel], m["row"].to_numpy()[sel], cur_code[sel]
        if name == "ambient":
            d = drawn_codes[s]
            # draw_local_ambient concatenates in granule order (all of granule d["granule"][0],
            # then [1], ...). `members` is ordered by transcript row, so scatter back through a
            # stable argsort on the group key.
            assert np.array_equal(np.unique(gs), np.sort(d["granule"])), (
                f"[{s}] the ambient granules drawn in section 3 are not the ones selected here")
            uniq, inv = np.unique(gs, return_inverse=True)
            order = np.argsort(inv, kind="stable")
            new = np.empty(len(gs), dtype=np.int32)
            new[order] = d["code"]
        else:
            # permute within granule: both orders group by granule identically, so this is a
            # within-granule shuffle of the labels and nothing else.
            src = np.lexsort((rng.random(len(gs)), gs))
            dst = np.lexsort((np.arange(len(gs)), gs))
            new = np.empty(len(gs), dtype=np.int32)
            new[dst] = cs[src]
        out_rows.append(rs)
        out_code.append(new)
        n_arm = int((arm == ARMS.index(name)).sum())
        audit.append(dict(sample=s, arm=name, n_granules_in_arm=n_arm,
                          n_granules_converted=int(np.unique(gs).size),
                          n_granules_dropped=n_arm - int(np.unique(gs).size),
                          n_transcripts=len(rs), n_changed=int((new != cs).sum()),
                          n_marker_before=int(np.isin(targets[cs], MARKERS).sum()),
                          n_marker_after=int(np.isin(targets[new], MARKERS).sum()),
                          pool_radius_um=RADIUS))

    patch = pd.DataFrame(dict(row=np.concatenate(out_rows).astype(np.int64),
                              new_target=pd.Categorical(targets[np.concatenate(out_code)],
                                                        categories=list(targets))))
    assert patch["row"].is_unique, f"[{s}] a transcript would be rewritten twice"
    A3.write_parquet_atomic(patch, C.pseudo_relabel_path(s))
    patches[s] = patch
    print(f"[{s}] patch: {len(patch):,} rows -> {C.pseudo_relabel_path(s).name}")

if audit:
    audit = pd.DataFrame(audit)
    audit.to_csv(OUT / "a3e_relabel_audit.csv", index=False)
    print()
    print(audit.to_string(index=False))

pd.DataFrame([dict(sample=s, n_transcripts=tx_len[s], n_relabelled=len(patches[s]),
                   pool_radius_um=RADIUS, min_pool=C.PSEUDO_MIN_POOL, frac=C.PSEUDO_FRAC,
                   seed=C.PSEUDO_SEED, n_granules=len(gran[s]),
                   n_retained=int(pool_keep[s].sum()),
                   n_thin_pool=int((~pool_keep[s]).sum()),
                   n_contaminated_untouched=len(contam[s]))
              for s in C.SAMPLES]).to_csv(OUT / "a3e_relabel_scope.csv", index=False)
print("\nwrote a3e_relabel_scope.csv -- run_pseudo_detection.py checks n_transcripts before "
      "applying anything")

[WT] read cached patch: 1,299,883 rows
[AD] read cached patch: 772,663 rows

wrote a3e_relabel_scope.csv -- run_pseudo_detection.py checks n_transcripts before applying anything


## 5. A local pre-check, before spending four hours on a node

mcDETECT runs one DBSCAN per marker gene and merges the results, so a granule is seeded exactly
when **some marker gene has a transcript inside the sphere with at least `min_samples` neighbours
of that same gene within ε**. `a3_common.dbscan_core_predicate` asks precisely that question with
two batched ball queries and no clustering, and A3b already relies on it.

Asking it once per marker and taking the union reproduces the seeding step of the detector, on the
relabelled coordinates, in minutes. It is not the whole pipeline — it does not merge, and it does
not apply the size, in-soma or negative-control filters, all of which can only remove spheres — so
it is an **upper bound on what the detection run will find**, and it is indicative rather than
authoritative. Its value is that it tells you what section 6 will say before the job is submitted.

Set `RUN_PRECHECK = False` to skip it: it builds one KD-tree per marker over roughly 30 M
transcripts and is the expensive local step in this notebook.

In [6]:
precheck = []
if RUN_PRECHECK:
    for s in C.SAMPLES:
        tx = A3.load_transcripts(s, columns=["global_x", "global_y", "global_z", "target"])
        # the relabelled table, in memory only -- nothing on disk is modified
        A3.apply_relabel_patch(tx, patches[s], sample=s, expect_rows=tx_len[s])

        rng = np.random.default_rng(zlib.crc32(f"{C.PSEUDO_SEED}|precheck|{s}".encode()))
        # the SAME population section 6 scores: granules that failed the retention rule were never
        # relabelled, so counting them in the ambient arm would dilute it with untouched granules,
        # and counting them in the control would compare against a different population.
        elig = pool_keep[s]
        unt = np.setdiff1d(np.flatnonzero((arms_of[s] == UNTOUCHED) & elig), contam[s])
        take = np.concatenate([np.flatnonzero((arms_of[s] != UNTOUCHED) & elig),
                               rng.choice(unt, size=min(20_000, len(unt)), replace=False)])
        sub = gran[s].iloc[take].reset_index(drop=True)
        sub["arm"] = np.asarray(ARMS)[arms_of[s][take]]

        by_gene = {}
        for gname in MARKERS:
            sel = tx["target"] == gname
            coords = tx.loc[sel, ["global_x", "global_y", "global_z"]].to_numpy(dtype=float)
            if len(coords):
                by_gene[gname] = (cKDTree(coords), coords)
        del tx

        hit = np.zeros(len(sub), dtype=bool)
        for gname in MARKERS:                      # union over the 20 per-gene DBSCAN runs
            sub["seed_gene"] = gname
            hit |= A3.dbscan_core_predicate(sub, by_gene)["would_detect"].to_numpy()
        del by_gene

        r = (pd.DataFrame(dict(arm=sub["arm"], seeded=hit))
             .groupby("arm").agg(n=("seeded", "size"), n_seeded=("seeded", "sum")).reset_index())
        r["rate"] = r["n_seeded"] / r["n"]
        r.insert(0, "sample", s)
        precheck.append(r)
        print(f"[{s}] would DBSCAN still seed a cluster here?")
        print(r.to_string(index=False), flush=True)

    precheck = pd.concat(precheck, ignore_index=True)
    precheck.to_csv(OUT / "a3e_precheck.csv", index=False)
else:
    print("RUN_PRECHECK is off -- section 6 is the authoritative answer")

[WT] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_WT_1/processed_data/transcripts.parquet
[WT] 103,398,068 transcripts
[WT] would DBSCAN still seed a cluster here?
sample       arm     n  n_seeded     rate
    WT   ambient 68073     13138 0.192999
    WT  scramble 68086     64867 0.952722
    WT untouched 20000     19993 0.999650
[AD] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_AD_1/processed_data/transcripts.parquet
[AD] 68,876,647 transcripts
[AD] would DBSCAN still seed a cluster here?
sample       arm     n  n_seeded     rate
    AD   ambient 39640      9004 0.227144
    AD  scramble 39616     37890 0.956432
    AD untouched 20000     19993 0.999650


## 6. The answer: what came back through the detector

Runs only once `output/a3e/detect_<sample>/spheres.parquet` exists. Until then this section reports
that it is waiting, and the notebook still completes.

### What "re-detected" means

**Credit follows the molecules, not proximity.** A granule counts as re-detected when the re-run
produced a sphere that

1. contains **at least half of that granule's own transcripts** — the granule-layer transcripts
   assigned to it in section 2 — and
2. contains **more of that granule's transcripts than of any other granule's**.

Stated plainly: *the detector rebuilt a sphere on the same transcripts, and that sphere is more
that granule than anything else.*

A purely geometric rule cannot say this. It cannot distinguish "the detector called this object
again" from "the detector called something else nearby", and here the distinction is everything:
80% of granules are untouched and will certainly be called, so a pseudo-granule whose contents were
entirely replaced can be credited to a neighbour. The size of that loophole is **measured, not
assumed** — matching the published granules against themselves, where a credit from any sphere
other than the granule's own is a credit it would collect for free:

| criterion | WT | AD |
|---|---|---|
| `center_in` | 6.98% | 8.20% |
| `intersect` | 33.91% | 38.98% |
| `merge` | 0.34% | 0.65% |
| condition 1 alone | 5.98% | 7.82% |
| **conditions 1 and 2** | **0.25%** | **0.28%** |

**Condition 2 is not cosmetic, and this is the one place the design was wrong before it was
measured.** Condition 1 alone was expected to solve the problem and does not: published granules
overlap so heavily that a neighbour's sphere already holds half of a given granule's transcripts
6–8% of the time, no better than `center_in`. Requiring the sphere to be *mostly* that granule —
one sphere is one object — is what buys the specificity, and it costs nothing: a granule is
credited by its own sphere 100.000% of the time at every threshold from 0.5 to 1.0.

The cell below regenerates this table from the data into `a3e_match_floor.csv`, so the choice rests
on the record rather than on this paragraph.

The three geometric rungs are **still scored and reported** beside the primary — they are mcDETECT's
own predicates (`a3_common.sphere_overlap`), so the answer can be read under every definition and
cannot be an artefact of ours:

- **`center_in`** — a new sphere's centre lies inside the original granule.
- **`intersect`** — the spheres touch at all. The loosest, and its floor is why it is not primary.
- **`merge`** — mcDETECT's own merge predicate, the strictest geometric rung.

The three arms are read together, and only together. `untouched` calibrates what re-detection means
at all; `scramble` says what the position scrambling costs by itself; `ambient` adds the change of
composition on top of it. The quantity that answers the reviewer is the gap between the last two.

In [ ]:
rate_rows, redet, floor_rows = [], {}, []
if all(have_detection.values()):
    for s in C.SAMPLES:
        new = pd.read_parquet(C.pseudo_detect_dir(s) / "spheres.parquet")
        print(f"[{s}] re-run produced {len(new):,} spheres against {len(gran[s]):,} published",
              flush=True)

        # the primary: did a new sphere rebuild on this granule's OWN transcripts?
        prov = A3.provenance_match(new, layer_xyz[s], owner_all[s], k_all[s], verbose=False)
        # the geometric ladder, unchanged, reported beside it
        geo = A3.match_spheres(gran[s], new, criteria=C.PSEUDO_MATCH_GEOMETRIC)
        mm = pd.concat([prov, geo], axis=1)
        redet[s] = mm

        # Two exclusions, both reported rather than folded into an arm. Precedence is thin pool
        # first: a granule whose neighbourhood could not supply the draw was never eligible for
        # any arm, whatever else is true of it. `not scorable` means it owns no transcripts, so
        # provenance is undefined for it.
        arm = np.asarray(ARMS)[arms_of[s]].astype(object)
        arm[contam[s]] = "excluded_contaminated"     # not silently counted as untouched
        arm[~pool_keep[s]] = "excluded_thin_pool"
        arm[~mm["scorable"].to_numpy()] = "excluded_no_transcripts"
        for crit in C.PSEUDO_MATCH_CRITERIA:
            t = (pd.DataFrame(dict(arm=arm, hit=mm[f"hit_{crit}"]))
                 .groupby("arm").agg(n=("hit", "size"), n_hit=("hit", "sum")).reset_index())
            t.insert(0, "criterion", crit)
            t.insert(0, "sample", s)
            rate_rows.append(t)

        # THE FALSE-CREDIT FLOOR, regenerated from data: the published granules matched against
        # THEMSELVES. Every granule matches itself, so a second match is a different granule
        # satisfying the rule -- the rate at which a destroyed pseudo-granule is credited for free.
        selfp = A3.provenance_match(gran[s], layer_xyz[s], owner_all[s], k_all[s], verbose=False)
        sc = selfp["scorable"].to_numpy()
        floor_rows.append(dict(sample=s, criterion="provenance", n_scored=int(sc.sum()),
                               false_credit=float((selfp["n_crediting"].to_numpy()[sc] > 1).mean()),
                               is_primary=True))
        selfg = A3.overlap_pairs(gran[s], gran[s], criterion=C.PSEUDO_MATCH_GEOMETRIC)
        for crit in C.PSEUDO_MATCH_GEOMETRIC:
            _, cnt = selfg[crit]
            floor_rows.append(dict(sample=s, criterion=crit, n_scored=len(gran[s]),
                                   false_credit=float((cnt > 1).mean()), is_primary=False))

    rate = pd.concat(rate_rows, ignore_index=True)
    rate["rate"] = rate["n_hit"] / rate["n"]
    rate["is_primary"] = rate["criterion"] == C.PSEUDO_MATCH_PRIMARY
    rate.to_csv(OUT / "a3e_redetection_rate.csv", index=False)

    floor = pd.DataFrame(floor_rows)
    floor.to_csv(OUT / "a3e_match_floor.csv", index=False)
    print("\nfalse-credit floor -- published granules matched against themselves:")
    print(floor.pivot_table(index="criterion", columns="sample", values="false_credit")
          .reindex(C.PSEUDO_MATCH_CRITERIA).to_string(float_format=lambda v: f"{v:.2%}"))

    print(f"\nre-detection, primary criterion ({C.PSEUDO_MATCH_PRIMARY}):")
    print(rate[rate["is_primary"]].to_string(index=False))
    for s in C.SAMPLES:
        q = rate[(rate["sample"] == s) & rate["is_primary"]].set_index("arm")["rate"]
        print(f"\n[{s}] untouched {q['untouched']:.1%} | own labels permuted "
              f"{q['scramble']:.1%} | drawn from local ambient {q['ambient']:.1%}"
              f"   -> the compositional effect alone is "
              f"{q['scramble'] - q['ambient']:+.1%}")

    print("\nsame arms under every criterion:")
    print(rate.pivot_table(index=["sample", "arm"], columns="criterion", values="rate")
          .reindex(columns=C.PSEUDO_MATCH_CRITERIA).to_string(float_format=lambda v: f"{v:.3f}"))
else:
    rate = None
    print("waiting for the detection run. Missing:",
          [s for s, v in have_detection.items() if not v])
    print("Copy output/a3e/ to HGCC, `sbatch slurm/run_pseudo_detection.sh`, rsync "
          "output/a3e/detect_*/ back, then re-run this notebook.")

### Why, mechanically

The re-detection rate is the result; this is the reason for it. DBSCAN needs `min_samples = 3`
transcripts of one marker gene within `eps = 1.5` µm. Every transcript keeps its position in all
three arms, so what changes is only how many of the transcripts in a sphere are markers at all.

In [8]:
shift = []
for s in C.SAMPLES:
    targets = np.asarray(all_targets[s])
    # Same population as sections 5 and 6: granules that failed the retention rule were never
    # relabelled, so leaving them in would show a slice of each converted arm carrying its
    # ORIGINAL marker count and would flatten the curve toward `untouched`.
    m = members[s]
    m = m[pool_keep[s][m["granule"].to_numpy()]]
    patch = patches[s].set_index("row")["new_target"].astype(str)
    before = np.isin(targets[m["cur_code"].to_numpy()], MARKERS)
    t_after = m["row"].map(patch)
    assert t_after.notna().all(), (
        f"[{s}] {int(t_after.isna().sum()):,} retained rows are missing from the patch")
    after = np.isin(t_after.to_numpy(), MARKERS)
    arm = np.asarray(ARMS)[arms_of[s][m["granule"].to_numpy()]]

    d = pd.DataFrame(dict(granule=m["granule"].to_numpy(), arm=arm,
                          before=before, after=after))
    per = d.groupby(["granule", "arm"], observed=True).agg(
        n_before=("before", "sum"), n_after=("after", "sum")).reset_index()

    # `untouched` here is the same granules BEFORE conversion -- the honest reference, since they
    # were drawn from the same population and are the only unperturbed counts available for them.
    curves = {"untouched": per["n_before"].to_numpy()}
    for a in C.PSEUDO_CONVERTED_ARMS:
        curves[a] = per.loc[per["arm"] == a, "n_after"].to_numpy()
    for a, v in curves.items():
        if not len(v):
            continue
        x = np.arange(0, int(np.quantile(v, 0.999)) + 2)
        shift.append(pd.DataFrame(dict(sample=s, arm=a, n_marker=x,
                                       frac=[(v <= k).mean() for k in x])))
    print(f"[{s}] marker transcripts per converted sphere: "
          f"{per['n_before'].median():.0f} before -> "
          + " | ".join(f"{a} {np.median(curves[a]):.0f}" for a in C.PSEUDO_CONVERTED_ARMS)
          + f"; below min_samples={C.DETECT_KWARGS_FINE['minspl']}: "
          + " | ".join(f"{a} {(curves[a] < C.DETECT_KWARGS_FINE['minspl']).mean():.1%}"
                       for a in ["untouched"] + list(C.PSEUDO_CONVERTED_ARMS)))

shift = pd.concat(shift, ignore_index=True)
shift.to_csv(OUT / "a3e_marker_shift.csv", index=False)

[WT] marker transcripts per converted sphere: 4 before -> ambient 2 | scramble 4; below min_samples=3: untouched 0.1% | ambient 51.0% | scramble 0.1%
[AD] marker transcripts per converted sphere: 4 before -> ambient 3 | scramble 4; below min_samples=3: untouched 0.2% | ambient 49.8% | scramble 0.2%


## 7. Correctness gates

**(d) is the load-bearing one.** If the untouched granules do not come back, either this re-run
does not reproduce the published pipeline or the perturbation did not stay local — and in either
case no other number in this notebook can be read. A3a's Set-2 reproduction rebuilt 681,346 spheres
against 681,337 published, and `miniball` is randomised at the 1e-13 level, so the threshold is set
just below 1 rather than at it.

**(g) checks the criterion itself.** The primary rule is only worth using if it does not hand out
credit for free, so (g) asserts that provenance's false-credit floor — measured by matching the
published granules against themselves — is below every geometric rung's. If it were not, the
criterion would be no better than the geometry it replaced.

**(f) checks that the arms are comparable.** `ambient` is read against `scramble`, so the two must
cover populations that differ in what was done to them and in nothing else. Gate (f) confirms the
retention rule left them matched in size, in granule transcript count, and in local pool size — if
the ambient arm were quietly restricted to granules in denser neighbourhoods, the comparison would
be measuring density rather than composition.

In [ ]:
if VALIDATE:
    # ---- (a) the arms are disjoint, exhaustive, and the right size ----
    for s in C.SAMPLES:
        a = arms_of[s]
        assert len(a) == len(gran[s])
        assert set(np.unique(a)) <= set(range(len(ARMS)))
        want = int(round(C.PSEUDO_FRAC * len(gran[s])))
        for name in C.PSEUDO_CONVERTED_ARMS:
            k = int((a == ARMS.index(name)).sum())
            assert k == want, f"(a) [{s}] {name} has {k} granules, expected {want}"
        assert int((a == UNTOUCHED).sum()) == len(gran[s]) - len(C.PSEUDO_CONVERTED_ARMS) * want
    print(f"[ok] (a) {len(C.PSEUDO_CONVERTED_ARMS)} converted arms of exactly "
          f"{C.PSEUDO_FRAC:.0%} each, disjoint, the rest untouched")

    # ---- (b) the patch rewrites each row once, and only granule-layer rows ----
    for s in C.SAMPLES:
        p_ = patches[s]
        assert p_["row"].is_unique, f"(b) [{s}] a transcript is rewritten twice"
        assert p_["row"].min() >= 0 and p_["row"].max() < tx_len[s]
        assert np.isin(p_["row"].to_numpy(), layer_rows[s]).all(), (
            f"(b) [{s}] the patch touches a row outside the granule layer -- somatic or residual "
            f"RNA would be rewritten")
        assert set(p_["new_target"].astype(str)) <= set(all_targets[s])
    print("[ok] (b) every rewritten row is a granule-layer transcript, rewritten exactly once, "
          "to a target that exists in this section")

    # ---- (b2) the retention rule actually held for every granule that was drawn ----
    for s in C.SAMPLES:
        k_of = np.bincount(members[s]["granule"].to_numpy(), minlength=len(gran[s]))
        drew = (arms_of[s] == ARMS.index("ambient")) & pool_keep[s] & (k_of > 0)
        assert (pool_size[s][drew] >= np.maximum(C.PSEUDO_MIN_POOL, k_of[drew])).all(), (
            f"(b2) [{s}] a granule was drawn from a pool smaller than max(min_pool, its own size)")
    print(f"[ok] (b2) every redrawn granule had pool >= max({C.PSEUDO_MIN_POOL}, its own "
          "transcript count), so every draw was without replacement")

    # ---- (c) `scramble` preserves each granule's composition EXACTLY ----
    for s in C.SAMPLES:
        m, targets = members[s], np.asarray(all_targets[s])
        pmap = patches[s].set_index("row")["new_target"].astype(str)
        # The population section 4 actually patched: granules failing the retention rule were
        # never rewritten, so scoring them here would compare the patch against a SUPERSET of
        # itself. `.map` would return NaN for their rows, groupby would silently drop those keys,
        # and the missing rows would surface as "composition changed" -- which is why the
        # membership assertion below comes first and says what is really wrong.
        gsel = m["granule"].to_numpy()
        sub = m[(arms_of[s][gsel] == ARMS.index("scramble")) & pool_keep[s][gsel]]
        t_after = sub["row"].map(pmap)
        assert t_after.notna().all(), (
            f"(c) [{s}] {int(t_after.isna().sum()):,} scramble rows are absent from the patch -- "
            f"this gate and section 4 are scoring different populations. That is NOT a "
            f"composition change; check the retention filter on both sides.")
        before = pd.DataFrame(dict(g=sub["granule"].to_numpy(),
                                   t=targets[sub["cur_code"].to_numpy()]))
        after = pd.DataFrame(dict(g=sub["granule"].to_numpy(), t=t_after.to_numpy()))
        cb = before.groupby(["g", "t"], observed=True).size().sort_index()
        ca = after.groupby(["g", "t"], observed=True).size().sort_index()
        assert cb.equals(ca), f"(c) [{s}] the scramble arm changed a granule's gene composition"
        assert (before["t"].to_numpy() != after["t"].to_numpy()).any(), (
            f"(c) [{s}] the scramble arm changed nothing at all")
    print("[ok] (c) the scramble arm permutes labels WITHIN each granule: every granule's gene "
          "multiset is identical before and after, so it differs from the ambient arm in "
          "composition and in nothing else")

    # ---- (f) the two converted arms are matched populations ----
    # Everything downstream reads `ambient` against `scramble`, so they have to differ in what was
    # done to them and in nothing else. The retention rule is applied to both for this reason.
    for s in C.SAMPLES:
        k_of = np.bincount(members[s]["granule"].to_numpy(), minlength=len(gran[s]))
        stats = {}
        for name in C.PSEUDO_CONVERTED_ARMS:
            sel = (arms_of[s] == ARMS.index(name)) & pool_keep[s] & (k_of > 0)
            stats[name] = dict(n=int(sel.sum()), k_med=float(np.median(k_of[sel])),
                               pool_med=float(np.median(pool_size[s][sel])),
                               r_med=float(np.median(gran[s]["sphere_r"].to_numpy()[sel])))
        a, b = (stats[n] for n in C.PSEUDO_CONVERTED_ARMS)
        assert abs(a["n"] - b["n"]) / max(a["n"], b["n"]) < 0.05, (
            f"(f) [{s}] arm sizes differ by more than 5%: {a['n']:,} vs {b['n']:,}")
        for key, tol in (("k_med", 0.25), ("pool_med", 0.10), ("r_med", 0.10)):
            d = abs(a[key] - b[key]) / max(a[key], b[key], 1e-9)
            assert d < tol, f"(f) [{s}] arms differ in {key} by {d:.1%}: {a[key]} vs {b[key]}"
        print(f"[ok] (f) [{s}] the two converted arms are matched: "
              + " | ".join(f"{n} n={stats[n]['n']:,} k={stats[n]['k_med']:.0f} "
                           f"pool={stats[n]['pool_med']:.0f}" for n in C.PSEUDO_CONVERTED_ARMS))

    # ---- (g) the primary criterion does not hand out credit for free ----
    if rate is not None:
        floor_w = (pd.read_csv(OUT / "a3e_match_floor.csv")
                   .pivot_table(index="sample", columns="criterion", values="false_credit"))
        for s in C.SAMPLES:
            fp = float(floor_w.loc[s, C.PSEUDO_MATCH_PRIMARY])
            # Asserted against center_in, the criterion provenance replaces. NOT against `merge`:
            # merge buys its low floor by being strict enough to miss genuine re-detections, so
            # beating it is not the bar -- being far better than the loose geometric rungs is.
            fc = float(floor_w.loc[s, "center_in"])
            assert fp < fc, (
                f"(g) [{s}] the primary criterion's false-credit floor ({fp:.2%}) is no better "
                f"than center_in's ({fc:.2%}) -- provenance is not buying the specificity it "
                f"exists for; check that the plurality condition is being applied")
            # a granule owning no transcripts has no provenance and must never be scored
            assert not (redet[s]["hit_provenance"] & ~redet[s]["scorable"]).any(), (
                f"(g) [{s}] a granule with no transcripts of its own was credited")
        print("[ok] (g) provenance's false-credit floor beats center_in's: "
              + ", ".join(f"{s} {float(floor_w.loc[s, C.PSEUDO_MATCH_PRIMARY]):.2%} vs "
                          f"center_in {float(floor_w.loc[s, 'center_in']):.2%}"
                          for s in C.SAMPLES))

    # ---- (d) THE CONTROL: untouched granules come back ----
    if rate is not None:
        prim = rate[rate["is_primary"]].set_index(["sample", "arm"])["rate"]
        for s in C.SAMPLES:
            u = float(prim.loc[(s, "untouched")])
            assert u >= C.PSEUDO_CONTROL_MIN, (
                f"(d) [{s}] only {u:.2%} of untouched granules were re-detected, below "
                f"{C.PSEUDO_CONTROL_MIN:.0%}. Either this run does not reproduce the published "
                f"pipeline or the perturbation was not local. Nothing else here is readable.")
        print(f"[ok] (d) untouched granules re-detected at "
              + ", ".join(f"{s} {float(prim.loc[(s,'untouched')]):.2%}" for s in C.SAMPLES)
              + " -- the re-run reproduces the published pipeline and the relabelling stayed local")

        # ---- (e) the answer does not depend on the matching criterion ----
        wide = rate.pivot_table(index=["sample", "arm"], columns="criterion", values="rate")
        wide = wide.reindex(columns=C.PSEUDO_MATCH_CRITERIA)
        print("\n[e] re-detection rate under each criterion:")
        print(wide.to_string())
        for s in C.SAMPLES:
            for crit in C.PSEUDO_MATCH_CRITERIA:
                assert wide.loc[(s, "ambient"), crit] <= wide.loc[(s, "untouched"), crit], (
                    f"(e) [{s}] under {crit} the ambient arm is re-detected at least as often as "
                    f"the untouched control")
        print("[ok] (e) the ordering holds under every criterion, loosest to strictest")
    else:
        print("[--] (d), (e) skipped: the detection run has not been read back yet")

## Outputs

| file | contents |
| --- | --- |
| `a3e_relabel_<sample>.parquet` | **the patch** — positional transcript row and its new target, for the rewritten rows only. This is what travels to HGCC |
| `a3e_pool_ladder.csv` | section 3: pool size and retention at each candidate radius, per sample — the check that 5 µm is a workable neighbourhood |
| `a3e_relabel_scope.csv` | per sample: transcript-table length (the guard `run_pseudo_detection.py` checks), granules, rewritten rows, retained vs thin-pool, radius and seed |
| `a3e_relabel_audit.csv` | per sample × arm: granules converted and dropped, transcripts rewritten, how many labels actually changed, marker counts before and after |
| `a3e_precheck.csv` | section 5: would DBSCAN still seed a cluster at each granule, by arm — indicative, computed locally before the job is submitted |
| `a3e_redetection_rate.csv` | **the result**: per sample × arm × criterion, granules re-detected by the full pipeline. `is_primary` marks `PSEUDO_MATCH_PRIMARY` (provenance) for the figure |
| `a3e_match_floor.csv` | the false-credit floor of every criterion, from matching the published granules against themselves — why provenance is the primary and `intersect` is not |
| `a3e_marker_shift.csv` | the mechanism: the distribution of marker transcripts per sphere, by arm |
| `detect_<sample>/` | written on HGCC by `run_pseudo_detection.py` — `spheres.parquet`, `sphere_dict.parquet`, `funnel_by_gene.csv`, `run_info.csv` |

Figures are drawn by `Rscript A3_figures.R` section 8, which writes
`output/figures/a3e_pseudo_granules.jpeg`.